In [1]:
# author: Wesley Chen
# For SSMIF project

# in v1, try 10y yield across country to predict spx, and sector returns, end up with all negative R^2 corrleation
# in v2 (this time), try US treasury yield with different maturities (yield curve), regress on sector returns

# get 3M, 2Y, 5Y, 10Y to represent the yield curve
import pandas as pd

files = [
    ("US_3M_yield.csv", "US_3M"),
    ("US_2Y_yield.csv", "US_2Y"),
    ("US_5Y_yield.csv", "US_5Y"),
    ("US_10Y_yield.csv", "US_10Y"),
]


def _read_yield_csv(fname, col_name):
    df = pd.read_csv(fname)
    if "observation_date" in df.columns:
        date_col = "observation_date"
    elif "DATE" in df.columns:
        date_col = "DATE"
    else:
        date_col = "Date"
    value_col = [c for c in df.columns if c != date_col][0]
    df = df.rename(columns={date_col: "date", value_col: col_name})
    df["date"] = pd.to_datetime(df["date"])
    return df[["date", col_name]]


df_curve = None
for fname, code in files:
    part = _read_yield_csv(fname, code)
    if df_curve is None:
        df_curve = part
    else:
        df_curve = pd.merge(df_curve, part, on="date", how="outer")

df_curve = df_curve.sort_values("date").reset_index(drop=True)
df_curve = df_curve.ffill()

yield_cols = [c for _, c in files]
df_curve = df_curve.dropna(subset=yield_cols)
df_curve = df_curve.loc[df_curve["date"] >= "2000-01-01"].reset_index(drop=True)

spx = pd.read_csv("SPX historical data.csv")
spx["date"] = pd.to_datetime(spx["Date"], errors="coerce").dt.normalize()
spx = spx[["date", "Close"]].rename(columns={"Close": "SPX_Close"})
df_curve["date"] = pd.to_datetime(df_curve["date"]).dt.normalize()

df = df_curve.merge(spx, on="date", how="inner").dropna(subset=["SPX_Close"]).reset_index(drop=True)

print(df.head(10))
print("...")
print(df.tail())
print(f"\ndf (yields + SPX) shape: {df.shape}")

# get the raw data of yield curve and spx price from Jan 2000 to Feb 2026 


        date  US_3M  US_2Y  US_5Y  US_10Y  SPX_Close
0 2000-01-03   5.27   6.38   6.50    6.58    1455.22
1 2000-01-04   5.27   6.30   6.40    6.49    1399.42
2 2000-01-05   5.28   6.38   6.51    6.62    1402.11
3 2000-01-06   5.25   6.35   6.46    6.57    1403.45
4 2000-01-07   5.22   6.31   6.42    6.52    1441.47
5 2000-01-10   5.24   6.38   6.49    6.57    1457.60
6 2000-01-11   5.27   6.45   6.57    6.67    1438.56
7 2000-01-12   5.29   6.49   6.63    6.72    1432.25
8 2000-01-13   5.25   6.40   6.54    6.63    1449.68
9 2000-01-14   5.25   6.44   6.59    6.69    1465.15
...
           date  US_3M  US_2Y  US_5Y  US_10Y  SPX_Close
6567 2026-02-12   3.61   3.47   3.67    4.09    6832.76
6568 2026-02-13   3.60   3.40   3.61    4.04    6836.17
6569 2026-02-17   3.61   3.43   3.63    4.05    6843.22
6570 2026-02-18   3.61   3.47   3.66    4.09    6881.31
6571 2026-02-19   3.61   3.47   3.65    4.08    6861.89

df (yields + SPX) shape: (6572, 6)


In [2]:
# load the 11 sector ETF prices (SP500 11 sectors); weekly log returns + lag1 (next week), same W-FRI convention as cell 2
# load sector data first, with try of regress on this data later
import os
import numpy as np

for _name in ("SP500 sectors 11.csv", "SP500_11_sectors.csv"):
    if os.path.isfile(_name):
        _sector_file = _name
        break
else:
    raise FileNotFoundError(
        "Expected 'SP500 sectors 11.csv' or 'SP500_11_sectors.csv' in the notebook working directory."
    )

df_sectors_raw = pd.read_csv(_sector_file)
df_sectors_raw["date"] = pd.to_datetime(df_sectors_raw["date"], errors="coerce").dt.normalize()
sector_cols = [c for c in df_sectors_raw.columns if c != "date"]

df_sectors_raw = df_sectors_raw.sort_values("date").dropna(subset=["date"])
df_sectors = (
    df_sectors_raw.set_index("date")[sector_cols]
    .sort_index()
    .resample("W-FRI")
    .last()
    .dropna(how="all")
    .reset_index()
)

# week-over-week log return; lag1 = next week's return (shift -1), aligned with SPX_log_ret_lag1 usage
for c in sector_cols:
    df_sectors[f"{c}_log_ret"] = np.log(df_sectors[c] / df_sectors[c].shift(1))
    df_sectors[f"{c}_log_ret_lag1"] = df_sectors[f"{c}_log_ret"].shift(-1)

print(f"Loaded {_sector_file} — weekly W-FRI (last in week), shape: {df_sectors.shape}")
print(f"Tickers: {sector_cols}")
print(df_sectors[["date"] + sector_cols[:3] + [f"{sector_cols[0]}_log_ret", f"{sector_cols[0]}_log_ret_lag1"]].head(6))


Loaded SP500_11_sectors.csv — weekly W-FRI (last in week), shape: (523, 34)
Tickers: ['XLC', 'XLRE', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY']
        date      XLC    XLRE        XLB  XLC_log_ret  XLC_log_ret_lag1
0 2015-10-09  56.1300  30.160  36.783966          NaN          0.000536
1 2015-10-16  56.1601  30.642  36.733990     0.000536          0.030511
2 2015-10-23  57.9000  31.070  37.508820     0.030511         -0.006298
3 2015-10-30  57.5365  31.146  37.675457    -0.006298          0.018315
4 2015-11-06  58.6000  29.950  37.767097     0.018315         -0.048423
5 2015-11-13  55.8300  29.560  36.992271    -0.048423          0.026513


In [3]:
# in this cell, process the data
# Weekly bars (W-FRI): last trading observation in each calendar week for yields and SPX.
# SPX log return and *_dbps are week-over-week (same definitions as before, on weekly rows).
import numpy as np

df = df.sort_values("date").reset_index(drop=True)

yield_cols = ["US_3M", "US_2Y", "US_5Y", "US_10Y"]
_agg = {c: "last" for c in yield_cols}
_agg["SPX_Close"] = "last"
df = (
    df.set_index("date")
    .sort_index()
    .resample("W-FRI")
    .agg(_agg)
    .dropna()
    .reset_index()
)

df["SPX_log_ret"] = np.log(df["SPX_Close"] / df["SPX_Close"].shift(1))
# Same-row week t: next week's log return log(P_{t+1}/P_t); pairs rates at t with SPX move to t+1
df["SPX_log_ret_lag1"] = df["SPX_log_ret"].shift(-1)

for col in yield_cols:
    df[f"{col}_dbps"] = (df[col] - df[col].shift(1)) * 100.0  # week-over-week, bps

# use 10y-2y, 10y-3m, 5y-2y to represent investor sentiment on economic growth
df["spread_10Y_2Y"] = df["US_10Y"] - df["US_2Y"]
df["spread_10Y_3M"] = df["US_10Y"] - df["US_3M"]
df["spread_5Y_2Y"] = df["US_5Y"] - df["US_2Y"]

spread_cols = ["spread_10Y_2Y", "spread_10Y_3M", "spread_5Y_2Y"]
for s_col in spread_cols:
    df[f"{s_col}_dbps"] = (df[s_col] - df[s_col].shift(1)) * 100.0  # week-over-week, bps

# 11 sector next-week log returns (cell 1), same W-FRI dates
_sector_lag_cols = [c for c in df_sectors.columns if c.endswith("_log_ret_lag1")]
df["date"] = pd.to_datetime(df["date"]).dt.normalize()
_sec = df_sectors[["date"] + _sector_lag_cols].copy()
_sec["date"] = pd.to_datetime(_sec["date"]).dt.normalize()
df = df.merge(_sec, on="date", how="left")

proc_cols = (
    ["date", "SPX_Close", "SPX_log_ret", "SPX_log_ret_lag1"]
    + yield_cols
    + [f"{c}_dbps" for c in yield_cols]
    + spread_cols
    + [f"{s}_dbps" for s in spread_cols]
    + _sector_lag_cols
)
print(f"df weekly shape: {df.shape}  (W-FRI, last obs in week)")
print(df[proc_cols].head(8))
print("...")
print(
    df[
        [
            "date",
            "SPX_log_ret",
            "SPX_log_ret_lag1",
            "spread_10Y_2Y_dbps",
            "spread_10Y_3M_dbps",
            "spread_5Y_2Y_dbps",
        ]
    ].tail(3)
)

df weekly shape: (1364, 29)  (W-FRI, last obs in week)
        date  SPX_Close  SPX_log_ret  SPX_log_ret_lag1  US_3M  US_2Y  US_5Y  \
0 2000-01-07    1441.47          NaN          0.016294   5.22   6.31   6.42   
1 2000-01-14    1465.15     0.016294         -0.016371   5.25   6.44   6.59   
2 2000-01-21    1441.36    -0.016371         -0.057985   5.31   6.48   6.67   
3 2000-01-28    1360.16    -0.057985          0.046127   5.48   6.58   6.68   
4 2000-02-04    1424.37     0.046127         -0.026500   5.50   6.63   6.64   
5 2000-02-11    1387.12    -0.026500         -0.030026   5.49   6.65   6.77   
6 2000-02-18    1346.09    -0.030026         -0.009502   5.58   6.66   6.72   
7 2000-02-25    1333.36    -0.009502          0.055299   5.62   6.45   6.50   

   US_10Y  US_3M_dbps  US_2Y_dbps  ...  XLRE_log_ret_lag1  XLB_log_ret_lag1  \
0    6.52         NaN         NaN  ...                NaN               NaN   
1    6.69         3.0        13.0  ...                NaN               NaN

In [4]:
# little try on sector data, will try other models later
# OLS: X = US yield curve (same as before), Y = each sector's next-week log return (11 targets)
import numpy as np
import pandas as pd
import statsmodels.api as sm

_yield_levels = ["US_3M", "US_2Y", "US_5Y", "US_10Y"]
_yield_dbps = [f"{c}_dbps" for c in _yield_levels]
_spread_levels = ["spread_10Y_2Y", "spread_10Y_3M", "spread_5Y_2Y"]
_spread_dbps = [f"{s}_dbps" for s in _spread_levels]
feature_cols = _yield_levels + _yield_dbps + _spread_levels + _spread_dbps

if "sector_cols" in globals() and sector_cols:
    target_cols = [f"{t}_log_ret_lag1" for t in sector_cols]
else:
    target_cols = [
        c
        for c in df.columns
        if c.endswith("_log_ret_lag1") and c != "SPX_log_ret_lag1"
    ]

missing = [c for c in feature_cols + target_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns (run prior cells): {missing}")

print("X — yield curve features (%d):" % len(feature_cols))
print(feature_cols)
print("\nY — sector targets (%d): next-week log return per ETF" % len(target_cols))
print(target_cols)

need_cols = feature_cols + target_cols
df_clean = df.dropna(subset=need_cols).sort_values("date").reset_index(drop=True)
print(f"\ndf_clean: {len(df_clean)} rows (complete cases for all X and all sector Y)")

train_frac = 0.8
n_train = int(len(df_clean) * train_frac)
train = df_clean.iloc[:n_train].copy()
test = df_clean.iloc[n_train:].copy()

X_train = sm.add_constant(train[feature_cols], has_constant="add")
X_test = sm.add_constant(test[feature_cols], has_constant="add")

ols_by_sector = {}
summary_rows = []
for tgt in target_cols:
    y_tr = train[tgt]
    y_te = test[tgt]
    ols = sm.OLS(y_tr, X_train).fit()
    ols_by_sector[tgt] = ols
    y_pred = ols.predict(X_test)
    rmse_te = float(np.sqrt(np.mean((y_te.values - y_pred.values) ** 2)))
    ss_res = float(((y_te - y_pred) ** 2).sum())
    ss_tot = float(((y_te - y_te.mean()) ** 2).sum())
    r2_te = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    tkr = tgt.replace("_log_ret_lag1", "")
    summary_rows.append(
        {
            "sector": tkr,
            "R2_train": float(ols.rsquared),
            "R2_test": r2_te,
            "RMSE_test": rmse_te,
        }
    )

ols_sector_table = pd.DataFrame(summary_rows)
print("\n" + "=" * 72)
print(f"OLS by sector — n_train={len(train)}, n_test={len(test)}")
print("=" * 72)
print(ols_sector_table.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n" + "=" * 72)
print(f"Example: full OLS summary for {target_cols[0]}")
print("=" * 72)
print(ols_by_sector[target_cols[0]].summary())

X — yield curve features (14):
['US_3M', 'US_2Y', 'US_5Y', 'US_10Y', 'US_3M_dbps', 'US_2Y_dbps', 'US_5Y_dbps', 'US_10Y_dbps', 'spread_10Y_2Y', 'spread_10Y_3M', 'spread_5Y_2Y', 'spread_10Y_2Y_dbps', 'spread_10Y_3M_dbps', 'spread_5Y_2Y_dbps']

Y — sector targets (11): next-week log return per ETF
['XLC_log_ret_lag1', 'XLRE_log_ret_lag1', 'XLB_log_ret_lag1', 'XLE_log_ret_lag1', 'XLF_log_ret_lag1', 'XLI_log_ret_lag1', 'XLK_log_ret_lag1', 'XLP_log_ret_lag1', 'XLU_log_ret_lag1', 'XLV_log_ret_lag1', 'XLY_log_ret_lag1']

df_clean: 522 rows (complete cases for all X and all sector Y)

OLS by sector — n_train=417, n_test=105
sector  R2_train   R2_test  RMSE_test
   XLC  0.072774 -0.305724   0.034000
  XLRE  0.055640 -0.058983   0.024203
   XLB  0.062942 -0.079028   0.023561
   XLE  0.060188 -0.171046   0.034288
   XLF  0.058094 -0.070100   0.024649
   XLI  0.072448 -0.094114   0.023742
   XLK  0.051580  0.023959   0.032600
   XLP  0.047249 -0.157013   0.014301
   XLU  0.072456 -0.341508   0.0230

In [5]:
# Simpler X: long yield, slope (10Y-2Y), 10Y-3M, and week-over-week changes (bps). Y: SPX next-week log return only (no sectors).
simple_feature_cols = [
    "US_10Y",
    "spread_10Y_2Y",
    "spread_10Y_3M",
    "US_10Y_dbps",
    "spread_10Y_2Y_dbps",
    "spread_10Y_3M_dbps",
]
simple_target_cols = ["SPX_log_ret_lag1"]

_keep = ["date"] + simple_target_cols + simple_feature_cols
df_simple = df[_keep].dropna().sort_values("date").reset_index(drop=True)
print(f"simple_feature_cols ({len(simple_feature_cols)}): {simple_feature_cols}")
print(f"simple_target_cols ({len(simple_target_cols)}): {simple_target_cols}")
print(f"df_simple shape: {df_simple.shape}")
print(df_simple.head())
print(df_simple.tail())

simple_feature_cols (6): ['US_10Y', 'spread_10Y_2Y', 'spread_10Y_3M', 'US_10Y_dbps', 'spread_10Y_2Y_dbps', 'spread_10Y_3M_dbps']
simple_target_cols (1): ['SPX_log_ret_lag1']
df_simple shape: (1362, 8)
        date  SPX_log_ret_lag1  US_10Y  spread_10Y_2Y  spread_10Y_3M  \
0 2000-01-14         -0.016371    6.69           0.25           1.44   
1 2000-01-21         -0.057985    6.79           0.31           1.48   
2 2000-01-28          0.046127    6.66           0.08           1.18   
3 2000-02-04         -0.026500    6.53          -0.10           1.03   
4 2000-02-11         -0.030026    6.63          -0.02           1.14   

   US_10Y_dbps  spread_10Y_2Y_dbps  spread_10Y_3M_dbps  
0         17.0                 4.0                14.0  
1         10.0                 6.0                 4.0  
2        -13.0               -23.0               -30.0  
3        -13.0               -18.0               -15.0  
4         10.0                 8.0                11.0  
           date  SPX_log

In [6]:
# try lasso, ridge, still first print summary of train model, then print summary of test model
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Use df_simple: same X; Y from simple_target_cols (SPX only when cell 4 is as set)
feature_cols = simple_feature_cols
df_clean = df_simple.dropna(subset=feature_cols + simple_target_cols).sort_values("date").reset_index(drop=True)
train_frac = 0.8
n_train = int(len(df_clean) * train_frac)
train = df_clean.iloc[:n_train]
test = df_clean.iloc[n_train:]

X_tr = train[feature_cols].to_numpy()
X_te = test[feature_cols].to_numpy()
print(
    f"Train/test from df_simple: n={len(df_clean)}, n_train={len(train)}, "
    f"features={len(feature_cols)}, targets={len(simple_target_cols)}"
)

_alphas = np.logspace(-4, 4, 40)

# create def of metrics to test on
def _directional_accuracy(y, y_hat):
    return float(np.mean(np.sign(y) == np.sign(y_hat)))

# pearson / spearman: undefined if y or y_hat is constant (flat predictions) — avoid scipy ConstantInputWarning
def _pearson_spearman(y, y_hat):
    y = np.asarray(y, dtype=float).ravel()
    y_hat = np.asarray(y_hat, dtype=float).ravel()
    if y.size < 2 or y_hat.size < 2:
        return (float("nan"), float("nan"))
    if np.std(y) < 1e-15 or np.std(y_hat) < 1e-15:
        return (float("nan"), float("nan"))
    pr, _ = pearsonr(y, y_hat)
    sr, _ = spearmanr(y, y_hat)
    return (float(pr) if np.isfinite(pr) else float("nan"), float(sr) if np.isfinite(sr) else float("nan"))


def _print_train_summary(title, pipe):
    y_hat_tr = pipe.predict(X_tr)
    rmse_tr = float(np.sqrt(mean_squared_error(y_tr, y_hat_tr)))
    r2_tr = float(r2_score(y_tr, y_hat_tr))
    dir_tr = _directional_accuracy(y_tr, y_hat_tr)
    pear_tr, spear_tr = _pearson_spearman(y_tr, y_hat_tr)
    print(
        f"{title} | train | RMSE={rmse_tr:.6f}  R^2={r2_tr:.6f}  Dir={dir_tr:.6f}  "
        f"Pearson r={pear_tr:.6f}  Spearman rho={spear_tr:.6f}"
    )


def _print_test_summary(title, pipe):
    y_hat_te = pipe.predict(X_te)
    rmse_te = float(np.sqrt(mean_squared_error(y_te, y_hat_te)))
    r2_te = float(r2_score(y_te, y_hat_te))
    dir_te = _directional_accuracy(y_te, y_hat_te)
    pear_te, spear_te = _pearson_spearman(y_te, y_hat_te)
    print(
        f"{title} | test  | RMSE={rmse_te:.6f}  R^2={r2_te:.6f}  Dir={dir_te:.6f}  "
        f"Pearson r={pear_te:.6f}  Spearman rho={spear_te:.6f}"
    )




Train/test from df_simple: n=1362, n_train=1089, features=6, targets=1


In [7]:
for i, tgt in enumerate(simple_target_cols):
    if i:
        print()
    tkr = tgt.replace("_log_ret_lag1", "")
    y_tr = train[tgt].to_numpy()
    y_te = test[tgt].to_numpy()

    ridge_pipe = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("reg", RidgeCV(alphas=_alphas, cv=5)),
        ]
    )
    ridge_pipe.fit(X_tr, y_tr)
    _print_train_summary(f"Ridge (RidgeCV) — {tkr}", ridge_pipe)
    _print_test_summary(f"Ridge (RidgeCV) — {tkr}", ridge_pipe)

    lasso_pipe = Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "reg",
                LassoCV(cv=5, random_state=0, max_iter=50_000),
            ),
        ]
    )
    lasso_pipe.fit(X_tr, y_tr)
    _print_train_summary(f"Lasso (LassoCV) — {tkr}", lasso_pipe)
    _print_test_summary(f"Lasso (LassoCV) — {tkr}", lasso_pipe)

# result: metrics vary by sector; R^2 often near 0 out-of-sample

Ridge (RidgeCV) — SPX | train | RMSE=0.025311  R^2=0.007251  Dir=0.560147  Pearson r=0.107125  Spearman rho=0.098463
Ridge (RidgeCV) — SPX | test  | RMSE=0.021999  R^2=0.001127  Dir=0.545788  Pearson r=0.079071  Spearman rho=0.089342
Lasso (LassoCV) — SPX | train | RMSE=0.025265  R^2=0.010866  Dir=0.557392  Pearson r=0.110348  Spearman rho=0.102182
Lasso (LassoCV) — SPX | test  | RMSE=0.022103  R^2=-0.008303  Dir=0.516484  Pearson r=0.021261  Spearman rho=0.047078


In [8]:
# Random forest: same X, each target y in simple_target_cols (requires cell 5 helpers + train/test + X_tr, X_te)
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

for i, tgt in enumerate(simple_target_cols):
    if i:
        print()
    tkr = tgt.replace("_log_ret_lag1", "")
    y_tr = train[tgt].to_numpy()
    y_te = test[tgt].to_numpy()

    rf_pipe = Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "reg",
                RandomForestRegressor(
                    n_estimators=800,
                    max_depth=4,
                    min_samples_leaf=100,
                    max_features="sqrt",
                    max_samples=0.85,
                    random_state=4253,
                    n_jobs=-1,
                ),
            ),
        ]
    )
    rf_pipe.fit(X_tr, y_tr)
    _print_train_summary(f"Random forest — {tkr}", rf_pipe)
    _print_test_summary(f"Random forest — {tkr}", rf_pipe)

Random forest — SPX | train | RMSE=0.025127  R^2=0.021671  Dir=0.563820  Pearson r=0.171333  Spearman rho=0.158750
Random forest — SPX | test  | RMSE=0.022060  R^2=-0.004442  Dir=0.553114  Pearson r=0.053202  Spearman rho=0.063045


In [9]:
# Calendar-quarter walk-forward (SPX): train on N full quarters of weekly rows, test on the next quarter only.
# Pattern (N=12): train 2015Q1–2017Q4 → test 2018Q1; train 2015Q2–2018Q1 → test 2018Q2; … (window slides one quarter).
# Requires df_simple + simple_feature_cols.
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

_FOCUS_Q = ["SPX"]
TRAIN_N_QUARTERS = 12  # quarters in training window (e.g. 12 ≈ 3 years)
_QUARTER_ALPHAS = np.logspace(-3, 4, 30)


def _spearman_q(y, y_hat):
    y = np.asarray(y, dtype=float).ravel()
    y_hat = np.asarray(y_hat, dtype=float).ravel()
    if y.size < 2 or np.std(y) < 1e-15 or np.std(y_hat) < 1e-15:
        return float("nan")
    r, _ = spearmanr(y, y_hat)
    return float(r) if np.isfinite(r) else float("nan")


summary_q = []
for tkr in _FOCUS_Q:
    tgt = "SPX_log_ret_lag1" if tkr == "SPX" else f"{tkr}_log_ret_lag1"
    sub = (
        df_simple[["date"] + simple_feature_cols + [tgt]]
        .dropna()
        .sort_values("date")
        .reset_index(drop=True)
    )
    sub["q"] = sub["date"].dt.to_period("Q")
    unique_q = sorted(sub["q"].unique())

    fold_rows = []
    for i in range(TRAIN_N_QUARTERS, len(unique_q)):
        train_qs = unique_q[i - TRAIN_N_QUARTERS : i]
        test_q = unique_q[i]
        tr = sub[sub["q"].isin(train_qs)]
        te = sub[sub["q"] == test_q]
        if len(tr) < 80 or len(te) < 3:
            continue

        X_train = tr[simple_feature_cols].to_numpy()
        y_train = tr[tgt].to_numpy()
        X_test = te[simple_feature_cols].to_numpy()
        y_test = te[tgt].to_numpy()

        pipe = Pipeline(
            [
                ("scaler", StandardScaler()),
                ("reg", RidgeCV(alphas=_QUARTER_ALPHAS, cv=5)),
            ]
        )
        pipe.fit(X_train, y_train)
        y_hat = pipe.predict(X_test)
        rho = _spearman_q(y_test, y_hat)
        r2 = float(r2_score(y_test, y_hat)) if len(y_test) >= 2 else float("nan")

        fold_rows.append(
            {
                "train_quarters": f"{train_qs[0]}–{train_qs[-1]}",
                "test_quarter": str(test_q),
                "n_train": len(tr),
                "n_test": len(te),
                "test_first_date": te["date"].min(),
                "test_last_date": te["date"].max(),
                "spearman": rho,
                "R2": r2,
            }
        )

    fq = pd.DataFrame(fold_rows)
    vr = fq["spearman"].replace([np.inf, -np.inf], np.nan).dropna()
    summary_q.append(
        {
            "ticker": tkr,
            "n_weeks": len(sub),
            "n_quarters_in_data": len(unique_q),
            "n_folds": len(fq),
            "spearman_mean": float(vr.mean()) if len(vr) else float("nan"),
            "spearman_median": float(vr.median()) if len(vr) else float("nan"),
            "spearman_std": float(vr.std(ddof=0)) if len(vr) else float("nan"),
            "pct_rho_pos": float((vr > 0).mean()) if len(vr) else float("nan"),
        }
    )

    print(f"\n=== {tkr} | quarter CV | train_window={TRAIN_N_QUARTERS} quarters ===")
    print(fq.to_string(index=False))
    print(
        f"Spearman (by quarter): mean={summary_q[-1]['spearman_mean']:.4f}  "
        f"median={summary_q[-1]['spearman_median']:.4f}  "
        f"std={summary_q[-1]['spearman_std']:.4f}  "
        f"pct_rho>0={summary_q[-1]['pct_rho_pos']:.2%}"
    )

print("\n--- Summary (calendar-quarter walk-forward, RidgeCV) ---")
print(pd.DataFrame(summary_q).to_string(index=False))



=== SPX | quarter CV | train_window=12 quarters ===
train_quarters test_quarter  n_train  n_test test_first_date test_last_date  spearman        R2
 2000Q1–2002Q4       2003Q1      155      13      2003-01-03     2003-03-28 -0.060440 -0.000973
 2000Q2–2003Q1       2003Q2      156      13      2003-04-04     2003-06-27 -0.390110 -0.609544
 2000Q3–2003Q2       2003Q3      156      13      2003-07-04     2003-09-26 -0.406593 -0.116655
 2000Q4–2003Q3       2003Q4      156      13      2003-10-03     2003-12-26 -0.153846 -0.387724
 2001Q1–2003Q4       2004Q1      156      13      2004-01-02     2004-03-26 -0.071429  0.008133
 2001Q2–2004Q1       2004Q2      156      13      2004-04-02     2004-06-25 -0.346154 -0.039487
 2001Q3–2004Q2       2004Q3      156      13      2004-07-02     2004-09-24  0.164835 -0.027868
 2001Q4–2004Q3       2004Q4      156      14      2004-10-01     2004-12-31  0.736264 -0.029703
 2002Q1–2004Q4       2005Q1      157      12      2005-01-07     2005-03-25  0.2447